# Annotated StockMixer

本 notebook 参照 AAAI 2024 论文 **StockMixer: A Simple Yet Strong MLP-Based Architecture for Stock Price Forecasting** 与官方代码实现，解释 StockMixer 如何用 MLP 风格的 mixing 结构完成股票收益预测。

论文链接：本目录 PDF [2024-AAAI-StockMixer.pdf](./2024-AAAI-StockMixer.pdf)  
官方代码：https://github.com/SJTU-Quant/StockMixer


## 1. 学习目标

本节围绕一个问题展开：**如何把一组股票的历史日频指标映射为下一交易日收益率预测，并用横截面排序指标评价模型是否有选股价值。**

学习完成后，你将能够：

- 说明 StockMixer 的输入 $X \in \mathbb{R}^{N\times T\times F}$ 在金融语境中的含义
- 区分 indicator mixing、time mixing、stock mixing 分别捕捉哪一类相关性
- 理解多尺度时间 patch 与上三角时间混合（TriU）为什么适合短窗口股票序列
- 写出 $\mathcal{L}_{\mathrm{MSE}}+\alpha\mathcal{L}_{\mathrm{rank}}$ 的训练目标，并解释它与选股排序的关系
- 用轻量示例验证模型前向传播、损失计算、训练循环和可视化


## 2. 研究背景与直觉

股票价格预测不是单只股票的孤立时间序列问题。论文把有效信息拆成三类相关性：

- **指标相关性（indicator correlation）**：同一天的开盘价、最高价、最低价、收盘价、均价等指标之间存在关系。例如收盘价相对开盘价的位置可能反映当日买卖压力。
- **时间相关性（temporal correlation）**：最近几个交易日的走势会影响下一日预期，但股票日频序列通常短、噪声大、非平稳，不像电力或交通数据那样有稳定周期。
- **股票相关性（stock correlation）**：同一市场中的股票受行业、风险偏好、流动性和市场状态共同影响。StockMixer 不依赖外部图，而是让模型学习“股票 -> 市场状态 -> 股票”的信息通路。

量化任务的目标不是只让价格数值误差最小，而是要让预测排序对选股有用。因此 notebook 会同时关注回归误差、横截面相关性和 top-k 股票的收益方向。


## 3. 任务定义

论文中的监督学习任务可以写成：

$$
X \in \mathbb{R}^{N\times T\times F}\ \xrightarrow{\theta}\ \hat{p}\in\mathbb{R}^{N\times 1}\ \rightarrow\ \hat{r}\in\mathbb{R}^{N\times 1}
$$

其中：

- $N$：同一市场中的股票数量，例如 NASDAQ 为 1026 只股票
- $T$：回看窗口长度，论文和官方代码使用 $T=16$ 个交易日
- $F$：每只股票每天的指标数，官方预处理数据中为 $F=5$
- $p_{i,t-1}$：第 $i$ 只股票在窗口最后一天的基准价格
- $r_{i,t}$：下一交易日真实收益率

官方训练代码先预测下一交易日价格 $\hat{p}_{i,t}$，再换算预测收益率：

$$
\hat{r}_{i,t}=\frac{\hat{p}_{i,t}-p_{i,t-1}}{p_{i,t-1}}
$$

在量化解释上，模型输出的 $\hat{r}$ 是一个横截面打分：每天对所有可交易股票排序，靠前股票被视为更值得买入或纳入组合。


## 4. 数据集介绍

论文使用三个美股日频数据集：NASDAQ、NYSE 和 S&P500。官方代码中 NASDAQ 预处理后的文件包括：

- `eod_data.pkl`：形状 `(1026, 1245, 5)`，即股票、交易日、指标
- `mask_data.pkl`：形状 `(1026, 1245)`，标记某只股票某天是否有效
- `gt_data.pkl`：形状 `(1026, 1245)`，下一期收益率标签
- `price_data.pkl`：形状 `(1026, 1245)`，用于从预测价格换算收益率

论文数据划分如下：

| 数据集 | 股票数 | 起始日期 | 结束日期 | 训练日 | 验证日 | 测试日 |
|---|---:|---|---|---:|---:|---:|
| NASDAQ | 1026 | 2013-01-02 | 2017-12-08 | 756 | 252 | 273 |
| NYSE | 1737 | 2013-01-02 | 2017-12-08 | 756 | 252 | 273 |
| S&P500 | 474 | 2016-01-04 | 2022-05-25 | 1006 | 253 | 352 |

本 notebook 的可运行示例直接加载参考代码仓库中的 NASDAQ 预处理数据，并默认使用全量 1026 只股票。训练回合数使用官方代码中的 `epochs = 100`。如果要严格复现论文表格，还应使用多次随机种子、相同评估协议和一致的运行环境，并保持时间顺序划分，避免未来信息泄漏。


## 5. 指标介绍

StockMixer 的评价关注“预测值是否能帮助选股”。常见指标包括：

- **MSE**：预测收益率与真实收益率的均方误差，衡量点预测是否接近真实收益。
- **IC（Information Coefficient）**：每天横截面上预测收益与真实收益的 Pearson 相关系数，再对多天取平均。IC 越高，说明预测分数越能解释真实收益。
- **RIC / Rank IC**：论文文字描述为基于排序的相关性，衡量股票短期收益潜力排序是否一致。官方代码中的 `RIC` 实现是 $\operatorname{mean}(IC) / \operatorname{std}(IC)$，更接近 IC 稳定性比率；复现实验时需要明确采用哪一种定义。
- **Precision@$N$**：取预测排名前 $N$ 的股票，统计真实收益为正的比例。例如 Precision@10=0.6 表示前 10 只中有 6 只下一期收益为正。
- **Sharpe Ratio**：用组合收益均值除以波动率，衡量收益与风险的权衡。官方代码中用 top-5 等权收益近似计算。

这些指标与金融验证路径对应：MSE 看回归精度，IC/Rank IC 看横截面排序，Precision@N 看多头选股命中率，Sharpe 看简单策略收益风险比。


## 6. 模型架构

StockMixer 的主线是：先在单只股票内部混合指标与时间信息，再在所有股票之间学习市场状态影响。下面的整体流程图描述金融任务视角：从历史窗口到个股表示，再到市场影响和预测收益率。

```mermaid
flowchart LR
    A[历史窗口 X: N x T x F] --> B[Conv1d 下采样]
    A --> C[MultiTime2dMixer]
    B --> C
    C --> D[channel_fc]
    D --> E[个股表示 y: N x d]
    E --> F[NoGraphStockMixer]
    E --> G[time_fc]
    F --> H[time_fc_market]
    G --> I[预测价格]
    H --> I
    I --> J[预测收益率与排序]
```

模型-组件树状图使用代码中的类名，便于把论文概念映射到实现：

```mermaid
flowchart TD
    S[StockMixer]
    S --> Conv[Conv1d]
    S --> MT[MultiTime2dMixer]
    S --> CFC[channel_fc: Linear]
    S --> NG[NoGraphStockMixer]
    S --> TFC[time_fc: Linear]
    S --> TFCM[time_fc_market: Linear]

    MT --> MMain[Mixer2dTriU: main_mixer]
    MT --> MScale[Mixer2dTriU: scale_mixer]

    MMain --> TriU1[TriU]
    MMain --> MB1[MixerBlock]
    MScale --> TriU2[TriU]
    MScale --> MB2[MixerBlock]

    NG --> LN[LayerNorm]
    NG --> D1[dense1: Linear N to m]
    NG --> ACT[Hardswish]
    NG --> D2[dense2: Linear m to N]
```

张量流可以概括为：

| 阶段/模块 | 模块类名 | 输入形状 | 操作 | 输出形状 | 金融含义 |
|---|---|---|---|---|---|
| 原始窗口 | 数据输入 | `(N, T, F)` | 取同一天、同一股票的多个指标 | `(N, T, F)` | 每只股票最近 $T$ 天的价格/技术指标 |
| 下采样时间尺度 | `Conv1d` | `(N, T, F)` | 沿时间维做 `kernel_size=2, stride=2` 的局部压缩 | `(N, T/2, F)` | 得到两日 patch 级别的短趋势 |
| time mixing | `TriU` | `(N, F, T)` 或 `(N, F, T/2)` | 只使用当前及过去时间点做因果时间混合 | `(N, F, T)` 或 `(N, F, T/2)` | 避免未来时间泄漏，提取短窗口趋势 |
| indicator mixing | `MixerBlock` | `(N, T, F)` 或 `(N, T/2, F)` | 在 $F$ 维上做两层 MLP mixing | `(N, T, F)` 或 `(N, T/2, F)` | 学习开高低收等指标间关系 |
| time + indicator block | `Mixer2dTriU` | `(N, T, F)` 或 `(N, T/2, F)` | `LayerNorm -> TriU -> 残差 -> MixerBlock -> 残差` | `(N, T, F)` 或 `(N, T/2, F)` | 单只股票内部先提取时间趋势，再融合指标 |
| multi-scale patch | `MultiTime2dMixer` | `(N, T, F)` 与 `(N, T/2, F)` | 拼接原始尺度、主尺度混合结果、下采样尺度混合结果 | `(N, 2T+T/2, F)` | 同时观察日级变化和局部趋势 |
| channel compression | `channel_fc` | `(N, 2T+T/2, F)` | 将指标维 $F$ 压缩到 1 | `(N, 2T+T/2)` | 得到每只股票的时间表示 |
| stock mixing | `NoGraphStockMixer` | `(N, d)` | 股票到市场状态再回到股票，$d=2T+T/2$ | `(N, d)` | 学习横截面市场影响 |
| own prediction head | `time_fc` | `(N, d)` | 对个股自身表示做线性预测 | `(N, 1)` | 个股自身历史模式给出的价格预测 |
| market prediction head | `time_fc_market` | `(N, d)` | 对市场影响表示做线性预测 | `(N, 1)` | 市场状态修正后的价格预测 |
| final prediction | `StockMixer.forward` | `(N, 1)` 与 `(N, 1)` | 两个预测分支相加 | `(N, 1)` | 输出下一期价格，进而转为收益 |


## 7. 模型组件

下面的代码来自官方 `model.py` 的教学化改写。保留核心结构，但把注释重点放在张量形状和金融含义上。


In [1]:
import math
import pickle
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(123456789)
torch.random.manual_seed(12345678)
random.seed(123456789)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

### MixerBlock模块

MixerBlock 相当于一个两层 MLP，主要用于指标通道混合。输入张量的最后一维记为 $D$，可以是指标维 $F$，也可以是其他需要 mixing 的表示维度。

```mermaid
flowchart LR
    A[输入 x: ..., D] --> B[Linear: D -> H]
    B --> C[GELU]
    C --> D{dropout > 0?}
    D -->|是| E[Dropout]
    D -->|否| F[Linear: H -> D]
    E --> F
    F --> G{dropout > 0?}
    G -->|是| H[Dropout]
    G -->|否| I[输出: ..., D]
    H --> I
```

在 StockMixer 中，`MixerBlock(channels)` 用于 indicator mixing：对每个股票、每个时间位置上的 $F$ 个价格指标做非线性组合，输出形状保持不变，便于后续残差连接。


In [ ]:
class MixerBlock(nn.Module):
    """在最后一个维度做两层 MLP mixing。

    输入形状: (..., D)
    输出形状: (..., D)
    """
    
    def __init__(self, dim, hidden_dim=None, dropout=0.0):
        super().__init__()
        hidden_dim = hidden_dim or dim
        self.dense_1 = nn.Linear(dim, hidden_dim)
        self.activation = nn.GELU()
        self.dense_2 = nn.Linear(hidden_dim, dim)
        self.dropout = dropout
        
    def forward(self, x):
        x = self.dense_1(x)
        x = self.activation(x)
        if self.dropout:
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.dense_2(x)
        if self.dropout:
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

### TriU模块

TriU 是因果时间混合模块。输入形状为 `(N, F, T)`：$N$ 是股票数，$F$ 是指标通道数，$T$ 是回看窗口长度。对当前时间 $t$ 的输出，TriU 只读取输入的 $0..t$ 历史前缀，不读取未来时间 $t+1..T-1$。

```mermaid
flowchart LR
    X["inputs: N x F x T"] --> P0["inputs[:, :, 0:1]"]
    X --> P1["inputs[:, :, 0:2]"]
    X --> P2["inputs[:, :, 0:3]"]
    X --> PM["..."]
    X --> PT["inputs[:, :, 0:t+1]"]

    P0 --> L0["Linear 1 -> 1"]
    P1 --> L1["Linear 2 -> 1"]
    P2 --> L2["Linear 3 -> 1"]
    PM --> LM["..."]
    PT --> LT["Linear t+1 -> 1"]

    L0 --> Y0["y[:, :, 0]"]
    L1 --> Y1["y[:, :, 1]"]
    L2 --> Y2["y[:, :, 2]"]
    LM --> YM["..."]
    LT --> YT["y[:, :, t]"]

    Y0 --> C["cat over time"]
    Y1 --> C
    Y2 --> C
    YM --> C
    YT --> C
    C --> Y["output: N x F x T"]
```

图中的 `...` 表示中间时间位置重复相同模式。

金融含义上，TriU 强制每个交易日位置只利用当日及之前的走势信息。这相当于给时间 mixing 加上因果约束，避免模型在时间维上看到未来交易日。


In [ ]:

class TriU(nn.Module):
    """因果时间混合：第 t 个输出只看 0..t 的历史。

    输入形状: (N, F, T)
    输出形状: (N, F, T)
    """

    def __init__(self, time_steps):
        super().__init__()
        self.time_steps = time_steps
        self.tri_u = nn.ModuleList([nn.Linear(i + 1, 1) for i in range(time_steps)])

    def forward(self, inputs):
        pieces = []
        for i, layer in enumerate(self.tri_u):
            # inputs[:, :, :i+1] 的最后一维只包含当前及过去交易日。
            pieces.append(layer(inputs[:, :, : i + 1]))
        return torch.cat(pieces, dim=-1)

### Mixer2dTriU模块

Mixer2dTriU 把因果时间混合和指标通道混合组合在一起。输入形状为 `(N, T, F)`，其中 $N$ 是股票数，$T$ 是回看窗口长度，$F$ 是每个交易日的指标数。

代码中的顺序是：先用 `TriU` 沿时间维做因果 time mixing，再用 `MixerBlock` 沿指标维做 indicator mixing。两处残差连接保证输出形状仍是 `(N, T, F)`。

```mermaid
flowchart LR
    A["inputs: N x T x F"] --> B["LayerNorm over T,F"]
    B --> C["permute -> N x F x T"]
    C --> D["TriU time mixing"]
    D --> E["permute -> N x T x F"]
    E --> R1["add original inputs"]
    A --> R1
    R1 --> LN2["LayerNorm over T,F"]
    LN2 --> M["MixerBlock on F"]
    M --> R2["add time mixed x"]
    LN2 --> R2
    R2 --> Y["output: N x T x F"]
```

金融含义上，这个模块先在每只股票内部沿交易日方向提取短期趋势，再在同一交易日内融合开高低收等指标。这样既保留了时间因果性，也让不同价格指标能够相互校正。


In [ ]:
class Mixer2dTriU(nn.Module):
    """先做时间 mixing，再做指标 mixing，并保留残差。

    输入形状: (N, T, F)
    输出形状: (N, T, F)
    """
    def __init__(self, time_steps, channels):
        super().__init__()
        self.ln_1 = nn.LayerNorm([time_steps, channels])
        self.ln_2 = nn.LayerNorm([time_steps, channels])
        self.time_mixer = TriU(time_steps)
        self.channel_mixer = MixerBlock(channels)

    def forward(self, inputs):
        x = self.ln_1(inputs)          # (N, T, F)
        x = x.permute(0, 2, 1)         # (N, F, T)
        x = self.time_mixer(x)         # (N, F, T)
        x = x.permute(0, 2, 1)         # (N, T, F)

        x = self.ln_2(x + inputs)      # 时间混合残差
        y = self.channel_mixer(x)      # 指标维混合
        return x + y

### MultiTime2dMixer模块

MultiTime2dMixer 用来把不同时间尺度的表示拼接起来。它接收两个输入：

- `inputs`：原始窗口，形状 `(N, T, F)`
- `scaled_inputs`：由前面的 `Conv1d(kernel_size=2, stride=2)` 得到的下采样窗口，形状 `(N, T/2, F)`

两个尺度分别经过 `Mixer2dTriU`，然后与原始输入沿时间维拼接。输出形状为 `(N, 2T+T/2, F)`。当 $T=16$ 时，时间长度为 $16+16+8=40$。

```mermaid
flowchart LR
    A["inputs: N x T x F"] --> M1["Mixer2dTriU on T"]
    B["scaled_inputs: N x T/2 x F"] --> M2["Mixer2dTriU on T/2"]

    A --> C["concat over time"]
    M1 --> C
    M2 --> C
    C --> Y["output: N x (2T + T/2) x F"]
```

金融含义上，原始尺度保留逐日价格指标，`Mixer2dTriU(inputs)` 提取因果短期趋势，`Mixer2dTriU(scaled_inputs)` 提取两日 patch 级别的更粗趋势。拼接后，模型可以同时利用日级变化和局部趋势，减少单一时间尺度对噪声的敏感性。


In [ ]:
class MultiTime2dMixer(nn.Module):
    """原始尺度 + 下采样尺度的时间表示拼接。

    对 T=16 而言，官方代码使用一次 stride=2 的卷积得到 8 个时间 patch，
    最终长度 d = 16 + 16 + 8 = 40。
    """

    def __init__(self, time_steps, channels, scale_dim=8):
        super().__init__()
        self.main_mixer = Mixer2dTriU(time_steps, channels)
        self.scale_mixer = Mixer2dTriU(scale_dim, channels)

    def forward(self, inputs, scaled_inputs):
        x = self.main_mixer(inputs)            # (N, T, F)
        y = self.scale_mixer(scaled_inputs)    # (N, T/2, F)
        return torch.cat([inputs, x, y], dim=1) # (N, 2T+T/2, F)


### NoGraphStockMixer模块

NoGraphStockMixer 用来建模股票之间的横截面关系，但不依赖行业图、知识图谱或超图先验。输入形状为 `(N, d)`，其中 $N$ 是股票数，$d$ 是每只股票经过时间与指标混合后的表示长度。

代码先把张量转成 `(d, N)`，这样每个表示维度都能在 $N$ 只股票之间做 mixing。随后用 `Linear(N -> m)` 把所有股票压缩成 $m$ 个市场状态，再用 `Linear(m -> N)` 把市场状态映射回每只股票。NASDAQ 配置中 $m=20$。

```mermaid
flowchart LR
    A["inputs: N x d"] --> P["permute -> d x N"]
    P --> LN["LayerNorm over N"]
    LN --> S2M["Linear N -> m"]
    S2M --> M["market states: d x m"]
    M --> Act["Hardswish"]
    Act --> M2S["Linear m -> N"]
    M2S --> Z["stock effects: d x N"]
    Z --> P2["permute -> N x d"]
    P2 --> Y["output: N x d"]
```

金融含义上，这相当于让模型先从全市场股票表示中提炼少数市场状态，例如风险偏好、行业共振或流动性环境，再把这些状态反馈到每只股票。它不是显式股票图，但能学习“股票 -> 市场 -> 股票”的横截面影响。


In [ ]:
class NoGraphStockMixer(nn.Module):
    """不使用外部图结构的股票混合。

    输入形状: (N, d)
    输出形状: (N, d)

    先把每个时间表示维度上的 N 只股票压缩到 m 个市场状态，
    再把 m 个市场状态映射回 N 只股票。
    """
    def __init__(self, stocks, market_states=20):
        super().__init__()
        self.layer_norm_stock = nn.LayerNorm(stocks)
        self.dense1 = nn.Linear(stocks, market_states)
        self.activation = nn.Hardswish()
        self.dense2 = nn.Linear(market_states, stocks)
    
    def forward(self, inputs):
        x = inputs.permute(1, 0)       # (d, N)
        x = self.layer_norm_stock(x)
        x = self.dense1(x)             # (d, m)
        x = self.activation(x)
        x = self.dense2(x)             # (d, N)
        return x.permute(1, 0)         # (N, d)
    

### StockMixer模块

如前文所述，`StockMixer` 将各个组件放在一起，完成从历史窗口到下一交易日价格预测的完整 forward 流程。输入为 `inputs`，形状是 `(N, T, F)`。

1. 首先通过 `conv: Conv1d` 得到下采样时间尺度：`(N, T, F) -> (N, T/2, F)`。这里的下采样相当于构造两日 patch 级别的短趋势。
2. 然后通过 `mixer: MultiTime2dMixer` 进行不同尺度上的时间与指标融合：原始窗口、主尺度混合结果、下采样尺度混合结果会沿时间维拼接，得到 `(N, 2T+T/2, F)`。
3. 接着通过 `channel_fc: Linear` 压缩指标维：`(N, 2T+T/2, F) -> (N, 2T+T/2)`，得到每只股票的时间表示 $y$。
4. 再通过 `stock_mixer: NoGraphStockMixer` 建模横截面市场影响：`y -> z`，其中 $z$ 的形状仍为 `(N, 2T+T/2)`。
5. 最后用两个预测头分别给出个股自身预测和市场影响预测：`time_fc(y)` 与 `time_fc_market(z)` 都输出 `(N, 1)`，二者相加得到最终预测价格。



金融含义上，`StockMixer` 先学习每只股票自身的多尺度历史模式，再学习同一市场中股票之间的共同状态，最后把个股自身信号和市场影响信号合并为下一交易日价格预测。


In [ ]:
class StockMixer(nn.Module):
    """StockMixer 教学版，接口与官方实现保持一致。"""

    def __init__(self, stocks, time_steps, channels, market_states=20):
        super().__init__()
        assert time_steps % 2 == 0, "本教学实现假设 time_steps 可以被 2 整除。"
        scale_dim = time_steps // 2
        mixed_time_dim = time_steps * 2 + scale_dim

        self.conv = nn.Conv1d(channels, channels, kernel_size=2, stride=2)
        self.mixer = MultiTime2dMixer(time_steps, channels, scale_dim=scale_dim)
        self.channel_fc = nn.Linear(channels, 1)
        self.stock_mixer = NoGraphStockMixer(stocks, market_states)
        self.time_fc = nn.Linear(mixed_time_dim, 1)
        self.time_fc_market = nn.Linear(mixed_time_dim, 1)

    def forward(self, inputs):
        # inputs: (N, T, F)
        scaled = self.conv(inputs.permute(0, 2, 1)).permute(0, 2, 1) # (N, T/2, F)
        y = self.mixer(inputs, scaled)                               # (N, 2T+T/2, F)
        y = self.channel_fc(y).squeeze(-1)                           # (N, 2T+T/2)

        z = self.stock_mixer(y)                                      # (N, 2T+T/2)
        own_price = self.time_fc(y)                                  # (N, 1)
        market_adjustment = self.time_fc_market(z)                   # (N, 1)
        return own_price + market_adjustment


### 7.1 先用小张量检查每一层形状

下面用 `N=6, T=16, F=5` 的小输入跑一次前向传播。真实 NASDAQ 的 `N=1026`，但教学验证不需要先加载完整市场。


In [6]:
N, T, F_DIM = 6, 16, 5
sample_x = torch.randn(N, T, F_DIM)
model_shape_check = StockMixer(stocks=N, time_steps=T, channels=F_DIM, market_states=3)

with torch.no_grad():
    scaled = model_shape_check.conv(sample_x.permute(0, 2, 1)).permute(0, 2, 1)
    mixed = model_shape_check.mixer(sample_x, scaled)
    embedding = model_shape_check.channel_fc(mixed).squeeze(-1)
    stock_effect = model_shape_check.stock_mixer(embedding)
    prediction = model_shape_check(sample_x)

shape_table = pd.DataFrame({
    "张量": ["sample_x", "scaled", "mixed", "embedding", "stock_effect", "prediction"],
    "形状": [tuple(t.shape) for t in [sample_x, scaled, mixed, embedding, stock_effect, prediction]],
    "含义": [
        "股票 x 回看窗口 x 指标",
        "两日 patch 后的短趋势序列",
        "原始尺度、时间通道混合原始尺度、时间通道混合patch尺度拼接",
        "每只股票的时间表示",
        "市场状态反馈后的股票表示",
        "下一交易日价格预测",
    ],
})
shape_table


,张量,形状,含义
0,sample_x,"(6, 16, 5)",股票 x 回看窗口 x 指标
1,scaled,"(6, 8, 5)",两日 patch 后的短趋势序列
2,mixed,"(6, 40, 5)",原始尺度、时间通道混合原始尺度、时间通道混合patch尺度拼接
3,embedding,"(6, 40)",每只股票的时间表示
4,stock_effect,"(6, 40)",市场状态反馈后的股票表示
5,prediction,"(6, 1)",下一交易日价格预测


## 8. 损失函数定义

官方训练目标由两部分组成：

$$
\mathcal{L}=\mathcal{L}_{\mathrm{MSE}}+\alpha\mathcal{L}_{\mathrm{rank}}
$$

回归项比较预测收益率与真实收益率：

$$
\mathcal{L}_{\mathrm{MSE}}=\frac{1}{|\Omega|}\sum_i m_i(\hat{r}_i-r_i)^2
$$

排序项惩罚横截面相对顺序错误：

$$
\mathcal{L}_{\mathrm{rank}}=\frac{1}{N^2}\sum_i\sum_j\max\left(0, -(\hat{r}_i-\hat{r}_j)(r_i-r_j)\right)
$$

如果真实收益 $r_i > r_j$，但模型预测 $\hat{r}_i < \hat{r}_j$，乘积为负，排序项就会产生惩罚。这个设计直接服务于选股：不仅要预测数值，还要把高收益股票排到前面。


In [7]:
def stockmixer_loss(predicted_price, ground_truth_return, base_price, mask, alpha=0.1):
    """计算 MSE + pairwise rank loss。

    predicted_price: (N, 1)
    ground_truth_return: (N, 1)
    base_price: (N, 1)
    mask: (N, 1)，1 表示该股票当天有效
    """
    predicted_return = (predicted_price - base_price) / base_price
    valid_count = mask.sum().clamp_min(1.0)
    reg_loss = ((predicted_return - ground_truth_return).pow(2) * mask).sum() / valid_count

    pred_diff = predicted_return - predicted_return.T       # (N, N)
    gt_diff = ground_truth_return - ground_truth_return.T   # (N, N)
    pair_mask = mask @ mask.T                               # (N, N)
    rank_loss = F.relu(-pred_diff * gt_diff * pair_mask).mean()
    loss = reg_loss + alpha * rank_loss
    return loss, reg_loss, rank_loss, predicted_return


## 9. 模型训练过程

完整官方训练流程是：

1. 按交易日顺序划分训练、验证、测试，避免未来信息泄漏。
2. 对每个 offset 取 `eod_data[:, offset:offset+T, :]`，得到所有股票同一天的横截面样本。
3. 用窗口最后一天价格 `price_data[:, offset+T-1]` 作为基准价。
4. 用 `gt_data[:, offset+T]` 作为下一期收益率标签。
5. 前向得到预测价格，换算成预测收益率。
6. 用 $\mathcal{L}_{\mathrm{MSE}}+\alpha\mathcal{L}_{\mathrm{rank}}$ 更新模型。
7. 每个 epoch 结束后计算验证集表现，并以验证集 loss 选择最佳模型。
8. 训练结束后加载最佳 checkpoint，只在测试集上报告论文对应的 5 个指标。

对照原始官方 `train.py` 可以看到：官方代码固定运行 `epochs = 100`，每轮都计算验证集和测试集，并在 `val_loss < best_valid_loss` 时更新 `best_valid_perf` 与 `best_test_perf`。它没有 `patience`、`break` 或类似 early stopping 的逻辑，因此严格说**没有早停机制**。本 notebook 保持这一点：仍然跑满设定 epoch，只是把验证集最优模型保存为 checkpoint，便于最后用同一组权重评估测试集。

下面的示例使用真实 NASDAQ 预处理数据，并采用全量股票、参考代码的训练/验证时间边界，以及官方代码中的 `epochs = 100`。这会比轻量教学验证耗时更久，但训练回合数已经和论文代码一致。

### 9.1 与官方代码的超参数对照

| 配置项 | 官方代码 | 本 notebook | 是否一致 | 说明 |
|---|---:|---:|---|---|
| market_name | NASDAQ | NASDAQ | 是 | 使用 NASDAQ 预处理数据 |
| stock_num | 1026 | 1026 | 是 | 全市场横截面，不是 mini-batch |
| lookback_length | 16 | 16 | 是 | 输入窗口 $T=16$ |
| fea_num | 5 | 5 | 是 | 每日指标数 $F=5$ |
| epochs | 100 | 100 | 是 | 官方训练回合数；不做 early stopping |
| valid_index | 756 | 756 | 是 | 验证集起点 |
| test_index | 1008 | 1008 | 是 | 测试集起点 |
| steps | 1 | 1 | 是 | 预测下一交易日 |
| learning_rate | 0.001 | 0.001 | 是 | Adam 学习率 |
| alpha | 0.1 | 0.1 | 是 | rank loss 权重 |
| market_num | 20 | 20 | 是 | `NoGraphStockMixer` 中市场状态数 $m$ |
| scale_factor | 3 | 未作为有效超参数使用 | 等价 | 参考代码传入 `scale=3`，但当前官方 `model.py` 的 `StockMixer` 未使用该参数 |
| activation | GELU | GELU | 是 | `MixerBlock` 使用 `nn.GELU()`；stock mixing 内部使用 `Hardswish`，与官方实现一致 |
| numpy seed | 123456789 | 123456789 | 是 | 对齐 `np.random.seed` |
| torch seed | 12345678 | 12345678 | 是 | 对齐 `torch.random.manual_seed` |
| batch_offsets | `np.arange(0, valid_index)` 后 shuffle | `np.arange(0, valid_index)` 后 shuffle | 是 | 每个 epoch 取前 `valid_index-lookback_length-steps+1` 个 offset |
| batch size | 无显式参数 | 无显式参数 | 是 | 每个 optimizer step 使用一个交易日 offset 的全量股票，张量为 `(1026, 16, 5)` |
| checkpoint | 无实际文件保存 | `stockmixer_best_valid.pt` | 增强 | 根据验证集 loss 保存最佳 `state_dict`，不改变训练目标 |

因此这里的“batch”不是样本 mini-batch，而是一个横截面训练样本：固定一个 `offset`，同时输入 1026 只股票过去 16 天的 5 个指标。


#### 数据集实现

官方 `load_data.py` 的预处理目标，是把逐股票 CSV 统一整理成四个矩阵。当前 notebook 直接读取已经保存好的 pkl 文件，但训练阶段使用的张量语义与官方代码一致。

官方 CSV 到训练矩阵的关键规则如下：

1. 每只股票读取一个日频文件，文件名形如 `NASDAQ_AAPL_1.csv`。
2. NASDAQ 数据会去掉最后一天，因为官方注释中说明最后一天缺失较多。
3. 原始 CSV 中 `-1234` 表示缺失值。若某天最后一列价格为 `-1234`，则 `mask=0`；否则 `mask=1`。
4. 真实收益率标签由最后一列价格计算：

$$
r_{i,t}=\frac{p_{i,t}-p_{i,t-steps}}{p_{i,t-steps}}
$$

5. 输入特征使用 `single_EOD[:, 1:]`，即跳过原始 CSV 第一列；基准价格 `price_data` 使用最后一列。
6. 对任意字段中的 `-1234`，官方代码用 `1.1` 替换，真正控制样本是否参与训练的是 `mask_data`。

```mermaid
flowchart LR
    A[逐股票 CSV] --> B[识别 -1234 缺失]
    B --> C[构造 mask_data]
    B --> D[替换缺失值为 1.1]
    D --> E[eod_data: N x D x F]
    D --> F[price_data: N x D]
    F --> G[gt_data: 收益率标签]
    E --> H[get_batch offset]
    F --> H
    G --> H
    C --> H
```

当前 notebook 中的 pkl 文件可以理解为上述流程的缓存结果。这样训练 notebook 不需要重复扫描大量 CSV，重点可以放在模型、loss 和验证流程上。

| 张量 | 形状 | 来源 | 金融含义 | 在训练中的作用 |
|---|---:|---|---|---|
| `eod_data` | `(N, D, F)` | `eod_data.pkl` | 每只股票每个交易日的 $F$ 个日频指标 | 输入窗口 `X[:, t:t+T, :]` |
| `mask_data` | `(N, D)` | `mask_data.pkl` | 股票在某交易日是否有有效行情 | 过滤停牌、缺失或异常样本 |
| `gt_data` | `(N, D)` | `gt_data.pkl` | 真实收益率标签 | 监督目标 $r_{i,t}$ |
| `price_data` | `(N, D)` | `price_data.pkl` | 基准价格 | 将预测价格换算成预测收益率 |

这里不使用 PyTorch `Dataset` 类，因为 StockMixer 的一个训练 step 不是抽若干只股票，而是抽一个交易日 `offset`，同时输入全市场股票。也就是说，每个样本是一个横截面面板：

$$
X_{offset}=\texttt{eod\_data}[:, offset:offset+T, :] \in \mathbb{R}^{N\times T\times F}
$$

标签来自窗口之后第 `steps` 天：

$$
y_{offset}=\texttt{gt\_data}[:, offset+T+steps-1] \in \mathbb{R}^{N\times 1}
$$

`mask` 使用 `offset` 到 `offset+T+steps-1` 这一段的最小值。只要某只股票在输入窗口或标签日有一天无效，该股票在这个横截面样本中就不参与 loss 和评价。这一步能减少停牌、缺失值或异常占位符造成的虚假横截面关系。

下面的实现分四步：定位数据目录、读取四个矩阵、复核官方预处理语义、用一个 `offset` 检查切片后的输入和标签形状。


In [ ]:
LOOKBACK_LENGTH = 16
PREDICT_STEPS = 1
VALID_INDEX = 756
TEST_INDEX = 1008


def find_nasdaq_dataset_dir():
    """定位 NASDAQ 预处理数据目录。"""
    candidates = [
        Path("dataset/NASDAQ"),
        Path("books/StockMixer/dataset/NASDAQ"),
    ]
    required = ["eod_data.pkl", "mask_data.pkl", "gt_data.pkl", "price_data.pkl"]
    for data_dir in candidates:
        if all((data_dir / name).exists() for name in required):
            return data_dir
    raise FileNotFoundError("未找到 NASDAQ 预处理数据，请检查 books/StockMixer/dataset/NASDAQ。")


def load_nasdaq_data(stock_subset=None):
    """加载 NASDAQ 真实预处理数据。

    stock_subset: 默认为 None，表示使用全部 1026 只股票；传入整数时可用于课堂调试子集。

    eod_data: (N, D, F)，股票、交易日、指标
    price_data: (N, D)，窗口最后一天基准价来自这里
    gt_data: (N, D)，第 t 天相对 t-1 天的真实收益率
    mask_data: (N, D)，1 表示该股票该交易日有效
    """
    data_dir = find_nasdaq_dataset_dir()
    with open(data_dir / "eod_data.pkl", "rb") as f:
        eod = pickle.load(f)
    with open(data_dir / "price_data.pkl", "rb") as f:
        price = pickle.load(f)
    with open(data_dir / "gt_data.pkl", "rb") as f:
        gt = pickle.load(f)
    with open(data_dir / "mask_data.pkl", "rb") as f:
        mask = pickle.load(f)

    if stock_subset is not None:
        eod = eod[:stock_subset]
        price = price[:stock_subset]
        gt = gt[:stock_subset]
        mask = mask[:stock_subset]

    return data_dir, eod.astype(np.float32), price.astype(np.float32), gt.astype(np.float32), mask.astype(np.float32)


def get_batch(offset, eod_data, price_data, gt_data, mask_data, lookback=16, steps=1):
    """取某个交易日 offset 对应的横截面训练样本。"""
    seq = eod_data[:, offset : offset + lookback, :]                         # (N, T, F)
    valid = mask_data[:, offset : offset + lookback + steps].min(axis=1)     # (N,)
    base_price = price_data[:, offset + lookback - 1]                        # (N,)
    label_return = gt_data[:, offset + lookback + steps - 1]                 # (N,)
    return (
        torch.tensor(seq, dtype=torch.float32, device=device),
        torch.tensor(base_price[:, None], dtype=torch.float32, device=device),
        torch.tensor(label_return[:, None], dtype=torch.float32, device=device),
        torch.tensor(valid[:, None], dtype=torch.float32, device=device),
    )


def describe_dataset_arrays(eod_data, price_data, gt_data, mask_data):
    """用表格检查四个预处理矩阵的形状和金融含义。"""
    return pd.DataFrame(
        [
            ["eod_data", tuple(eod_data.shape), "股票 x 交易日 x 指标", "模型输入窗口"],
            ["price_data", tuple(price_data.shape), "股票 x 交易日", "预测价格转收益率的基准价"],
            ["gt_data", tuple(gt_data.shape), "股票 x 交易日", "真实收益率标签"],
            ["mask_data", tuple(mask_data.shape), "股票 x 交易日", "有效样本掩码"],
        ],
        columns=["张量", "形状", "维度语义", "训练作用"],
    )


def describe_time_splits(total_days, valid_index=VALID_INDEX, test_index=TEST_INDEX):
    """说明按时间顺序切分后的交易日边界。"""
    return pd.DataFrame(
        [
            ["train", f"[0, {valid_index})", valid_index, "参与梯度更新"],
            ["valid", f"[{valid_index}, {test_index})", test_index - valid_index, "选择最佳 checkpoint"],
            ["test", f"[{test_index}, {total_days})", total_days - test_index, "最终样本外报告"],
        ],
        columns=["区间", "交易日索引", "天数", "用途"],
    )


def describe_official_preprocess():
    """把官方 load_data.py 的核心处理规则整理成表格。"""
    return pd.DataFrame(
        [
            ["读取单股票 CSV", "np.genfromtxt(..., dtype=np.float32)", "每只股票形成一个 (D, raw_cols) 日频矩阵"],
            ["NASDAQ 去尾", "single_EOD = single_EOD[:-1, :]", "删除缺失较多的最后一个交易日"],
            ["缺失识别", "最后一列价格等于 -1234 时 mask=0", "该股票该日不参与有效样本"],
            ["标签计算", "(p_t - p_{t-steps}) / p_{t-steps}", "下一期收益率监督信号"],
            ["缺失替换", "所有 -1234 替换为 1.1", "避免数值占位符直接进入模型"],
            ["输入矩阵", "eod_data[index, :, :] = single_EOD[:, 1:]", "保留日频指标作为 (N, D, F) 输入"],
            ["基准价格", "base_price[index, :] = single_EOD[:, -1]", "用于把预测价格换算成预测收益率"],
        ],
        columns=["步骤", "官方代码逻辑", "金融/建模含义"],
    )


def check_label_alignment(price_data, gt_data, mask_data, steps=1, day=VALID_INDEX):
    """抽查 gt_data 是否等于 price_data 的 steps 日收益率。"""
    previous_day = day - steps
    valid = (mask_data[:, day] > 0.5) & (mask_data[:, previous_day] > 0.5)
    computed_return = (price_data[valid, day] - price_data[valid, previous_day]) / price_data[valid, previous_day]
    label_return = gt_data[valid, day]
    abs_error = np.abs(computed_return - label_return)
    return pd.DataFrame(
        [
            [int(valid.sum()), float(np.nanmean(abs_error)), float(np.nanmax(abs_error))],
        ],
        columns=["有效股票数", "平均绝对误差", "最大绝对误差"],
    )


def describe_batch(offset, eod_data, price_data, gt_data, mask_data, lookback=16, steps=1):
    """展示一个 offset 如何映射成训练所需的四个张量。"""
    x, base_price, gt_return, valid_mask = get_batch(
        offset, eod_data, price_data, gt_data, mask_data, lookback, steps
    )
    return pd.DataFrame(
        [
            ["x", tuple(x.shape), f"eod_data[:, {offset}:{offset + lookback}, :]", "历史窗口，输入模型"],
            ["base_price", tuple(base_price.shape), f"price_data[:, {offset + lookback - 1}]", "窗口最后一天基准价"],
            ["gt_return", tuple(gt_return.shape), f"gt_data[:, {offset + lookback + steps - 1}]", "下一期真实收益率"],
            ["valid_mask", tuple(valid_mask.shape), f"min(mask_data[:, {offset}:{offset + lookback + steps}])", "过滤窗口或标签日无效股票"],
        ],
        columns=["张量", "形状", "索引来源", "金融含义"],
    )


data_dir, eod_data, price_data, gt_data, mask_data = load_nasdaq_data(stock_subset=None)
print("data_dir", data_dir)
print("mask coverage", float(mask_data.mean()))

official_preprocess_summary = describe_official_preprocess()
dataset_summary = describe_dataset_arrays(eod_data, price_data, gt_data, mask_data)
split_summary = describe_time_splits(mask_data.shape[1])
label_alignment_summary = check_label_alignment(price_data, gt_data, mask_data, PREDICT_STEPS, VALID_INDEX)
sample_offset = VALID_INDEX - LOOKBACK_LENGTH - PREDICT_STEPS + 1
batch_summary = describe_batch(sample_offset, eod_data, price_data, gt_data, mask_data, LOOKBACK_LENGTH, PREDICT_STEPS)

official_preprocess_summary, dataset_summary, split_summary, label_alignment_summary, batch_summary


#### 模型评估实现

StockMixer 的评估不是逐股票时间序列误差，而是逐交易日做横截面评价。给定一组连续 offsets，模型会输出三个矩阵：

| 矩阵 | 形状 | 维度语义 | 说明 |
|---|---:|---|---|
| `prediction` | `(N, num_days)` | 股票 x 评估交易日 | 模型预测收益率 |
| `ground_truth` | `(N, num_days)` | 股票 x 评估交易日 | 真实收益率 |
| `mask` | `(N, num_days)` | 股票 x 评估交易日 | 该股票当天是否有效 |

评估时每一列是一日横截面。也就是说，IC、Precision@10 和 Sharpe 都先在同一天的股票池中排序或计算收益，再跨交易日取平均。

```mermaid
flowchart LR
    A[offsets] --> B[get_batch]
    B --> C[model 预测价格]
    C --> D[换算预测收益率]
    D --> E[prediction: N x num_days]
    B --> F[ground_truth: N x num_days]
    B --> G[mask: N x num_days]
    E --> H[横截面指标]
    F --> H
    G --> H
    H --> I[mse / IC / RIC / prec_10 / sharpe5]
```

这里保留官方 `evaluator.py` 的 5 个指标名称和数量：

| 指标 | 官方键名 | 金融含义 | 需要注意的口径 |
|---|---|---|---|
| 均方误差 | `mse` | 预测收益率与真实收益率的点预测误差 | 用 `mask` 加权，只统计有效股票 |
| 信息系数 | `IC` | 每天预测收益与真实收益的横截面 Pearson 相关均值 | 反映打分与真实收益的线性相关性 |
| IC 稳定性比率 | `RIC` | 官方实现为 `mean(IC) / std(IC)` | 不是严格 Spearman Rank IC，需在复现实验中说明 |
| Top-10 命中率 | `prec_10` | 预测排名前 10 股票中真实收益非负的比例 | 衡量多头候选池方向正确率 |
| Top-5 Sharpe | `sharpe5` | 预测排名前 5 股票等权收益的年化风险调整表现 | 官方使用 `15.87`，约等于 `sqrt(252)` |

教学实现比官方代码多做两点防护：第一，top-k 选择只在 `mask > 0.5` 的股票中进行；第二，当有效样本或标准差不足时返回 `np.nan`，避免除零导致误读。


In [ ]:
def describe_evaluation_metrics():
    """说明 StockMixer 官方评估指标的名称、形状和金融含义。"""
    return pd.DataFrame(
        [
            ["mse", "标量", "有效股票上的均方误差", "点预测误差，越低越好"],
            ["IC", "标量", "逐日横截面 Pearson 相关后取均值", "预测分数解释真实收益的能力，通常越高越好"],
            ["RIC", "标量", "mean(IC) / std(IC)", "官方口径下的 IC 稳定性比率，不是严格 Spearman Rank IC"],
            ["prec_10", "标量", "预测 top-10 股票真实收益非负比例", "多头候选池方向命中率，越高越好"],
            ["sharpe5", "标量", "预测 top-5 等权收益的年化 Sharpe", "简单多头组合收益风险比，未扣手续费和滑点"],
        ],
        columns=["指标", "输出形状", "计算方式", "金融解释"],
    )


def select_top_valid_indices(scores, valid_mask, top_k):
    """在有效股票中按分数从高到低选择 top-k。"""
    valid_indices = np.flatnonzero(valid_mask > 0.5)
    if valid_indices.size < top_k:
        return np.array([], dtype=int)
    valid_scores = scores[valid_indices]
    order = np.argsort(valid_scores)[-top_k:]
    return valid_indices[order]


def predict_offsets(model, offsets, eod_data, price_data, gt_data, mask_data, lookback=16, steps=1, alpha=0.1):
    """逐日预测横截面收益，返回形状均为 (N, num_days) 的矩阵。"""
    model.eval()
    pred_list, gt_list, mask_list = [], [], []
    with torch.no_grad():
        for offset in offsets:
            x, base_price, gt_return, valid_mask = get_batch(
                offset, eod_data, price_data, gt_data, mask_data, lookback, steps
            )
            predicted_price = model(x)
            _, _, _, predicted_return = stockmixer_loss(
                predicted_price, gt_return, base_price, valid_mask, alpha=alpha
            )
            pred_list.append(predicted_return.squeeze(-1).cpu().numpy())
            gt_list.append(gt_return.squeeze(-1).cpu().numpy())
            mask_list.append(valid_mask.squeeze(-1).cpu().numpy())
    return np.stack(pred_list, axis=1), np.stack(gt_list, axis=1), np.stack(mask_list, axis=1)


def evaluate_stockmixer_metrics(prediction, ground_truth, mask):
    """对齐官方 evaluator.py 和论文表格的 5 个指标。

    prediction: (N, num_days)，预测收益率
    ground_truth: (N, num_days)，真实收益率
    mask: (N, num_days)，1 表示该股票该评估日有效
    """
    prediction = np.asarray(prediction, dtype=float)
    ground_truth = np.asarray(ground_truth, dtype=float)
    mask = np.asarray(mask, dtype=float)
    assert prediction.shape == ground_truth.shape == mask.shape, "shape mis-match"

    valid_count = np.maximum(mask.sum(), 1.0)
    mse = np.linalg.norm((prediction - ground_truth) * mask) ** 2 / valid_count

    # 与官方 evaluator.py 保持一致：先把无效位置乘为 0，再逐日计算相关系数。
    df_pred = pd.DataFrame(prediction * mask)
    df_gt = pd.DataFrame(ground_truth * mask)

    ic_values = []
    precision_10_values = []
    top5_returns = []

    for day in range(prediction.shape[1]):
        ic_values.append(df_pred[day].corr(df_gt[day]))

        top5 = select_top_valid_indices(prediction[:, day], mask[:, day], top_k=5)
        top10 = select_top_valid_indices(prediction[:, day], mask[:, day], top_k=10)

        if top5.size == 5:
            top5_returns.append(float(np.mean(ground_truth[top5, day])))
        if top10.size == 10:
            precision_10_values.append(float(np.mean(ground_truth[top10, day] >= 0)))

    ic_values = np.asarray(ic_values, dtype=float)
    top5_returns = np.asarray(top5_returns, dtype=float)
    precision_10_values = np.asarray(precision_10_values, dtype=float)

    ic_mean = float(np.nanmean(ic_values)) if ic_values.size else np.nan
    ic_std = float(np.nanstd(ic_values)) if ic_values.size else np.nan
    sharpe_std = float(np.nanstd(top5_returns)) if top5_returns.size else np.nan

    return {
        "mse": float(mse),
        "IC": ic_mean,
        "RIC": ic_mean / ic_std if ic_std > 1e-12 else np.nan,
        "prec_10": float(np.nanmean(precision_10_values)) if precision_10_values.size else np.nan,
        "sharpe5": float(np.nanmean(top5_returns) / sharpe_std * 15.87) if sharpe_std > 1e-12 else np.nan,
    }


def validate_offsets(model, offsets, eod_data, price_data, gt_data, mask_data, lookback=16, steps=1, alpha=0.1):
    """计算指定时间段的平均 loss 和论文指标。"""
    model.eval()
    loss_values, reg_values, rank_values = [], [], []
    pred_list, gt_list, mask_list = [], [], []

    with torch.no_grad():
        for offset in offsets:
            x, base_price, gt_return, valid_mask = get_batch(
                offset, eod_data, price_data, gt_data, mask_data, lookback, steps
            )
            predicted_price = model(x)
            loss, reg_loss, rank_loss, predicted_return = stockmixer_loss(
                predicted_price, gt_return, base_price, valid_mask, alpha=alpha
            )
            loss_values.append(loss.item())
            reg_values.append(reg_loss.item())
            rank_values.append(rank_loss.item())
            pred_list.append(predicted_return.squeeze(-1).cpu().numpy())
            gt_list.append(gt_return.squeeze(-1).cpu().numpy())
            mask_list.append(valid_mask.squeeze(-1).cpu().numpy())

    prediction = np.stack(pred_list, axis=1)
    ground_truth = np.stack(gt_list, axis=1)
    valid_mask = np.stack(mask_list, axis=1)
    metrics = evaluate_stockmixer_metrics(prediction, ground_truth, valid_mask)
    return {
        "loss": float(np.mean(loss_values)),
        "reg_loss": float(np.mean(reg_values)),
        "rank_loss": float(np.mean(rank_values)),
        "metrics": metrics,
        "prediction": prediction,
        "ground_truth": ground_truth,
        "mask": valid_mask,
    }


def format_stockmixer_metrics(metrics):
    return (
        f"mse:{metrics['mse']:.2e}, "
        f"IC:{metrics['IC']:.2e}, "
        f"RIC:{metrics['RIC']:.2e}, "
        f"prec@10:{metrics['prec_10']:.2e}, "
        f"SR:{metrics['sharpe5']:.2e}"
    )


toy_prediction = np.array([
    [0.10, 0.03, -0.02],
    [0.08, 0.01, 0.04],
    [0.05, -0.02, 0.03],
    [0.03, 0.04, 0.01],
    [0.01, 0.02, -0.01],
    [-0.01, 0.05, 0.02],
    [-0.02, -0.01, 0.00],
    [0.06, 0.00, 0.05],
    [0.02, 0.06, -0.03],
    [0.04, 0.07, 0.06],
    [0.07, 0.08, 0.07],
    [0.09, 0.09, 0.08],
], dtype=float)
toy_ground_truth = np.array([
    [0.06, 0.01, -0.01],
    [0.05, -0.02, 0.02],
    [0.04, -0.01, 0.01],
    [0.02, 0.03, -0.02],
    [-0.01, 0.01, -0.03],
    [-0.02, 0.04, 0.00],
    [-0.03, -0.02, -0.01],
    [0.03, 0.00, 0.04],
    [0.01, 0.05, -0.04],
    [0.02, 0.06, 0.05],
    [0.04, 0.07, 0.06],
    [0.05, 0.08, 0.07],
], dtype=float)
toy_mask = np.ones_like(toy_prediction)
toy_mask[6, 1] = 0.0

toy_metrics = evaluate_stockmixer_metrics(toy_prediction, toy_ground_truth, toy_mask)
display(describe_evaluation_metrics())
pd.Series(toy_metrics).to_frame("toy_value")


#### 模型训练实现

训练循环要回答三个问题：

1. **训练样本从哪里来**：每个 optimizer step 随机取一个训练期 `offset`，输入形状为 `(N, T, F)` 的全市场横截面窗口。
2. **优化目标是什么**：模型先预测下一期价格，再用 `base_price` 换算成预测收益率，优化 `MSE + alpha * rank_loss`。
3. **如何选模型**：每个 epoch 后在验证集 offsets 上计算平均 loss；如果 `val_loss` 更低，就保存 `stockmixer_best_valid.pt`。

官方代码固定训练 `epochs = 100`，并没有 early stopping。本 notebook 延续这一点：验证集只用于选择最佳 checkpoint，不用于提前停止训练。

```mermaid
flowchart TD
    A[初始化模型和 Adam] --> B[构造 train offsets]
    B --> C[每个 epoch 打乱 train offsets]
    C --> D[get_batch: N x T x F]
    D --> E[StockMixer 预测价格]
    E --> F[换算预测收益率]
    F --> G[MSE + alpha rank loss]
    G --> H[反向传播和参数更新]
    H --> I[验证集 validate_offsets]
    I --> J{val_loss 是否更低}
    J -->|是| K[保存最佳 checkpoint]
    J -->|否| L[只记录 history]
    K --> M[继续直到 100 epoch]
    L --> M
```

训练 offset 的边界需要特别小心。为了预测 `offset + T + steps - 1`，训练期最后一个可用 offset 是 `valid_index - T - steps`。验证集和测试集也采用同样的窗口逻辑向前回看，因此验证 offsets 从 `valid_index - T - steps + 1` 开始。这不是未来泄漏，而是用验证期第一天标签所需的历史窗口。


In [ ]:
stocks, lookback, features = eod_data.shape[0], LOOKBACK_LENGTH, eod_data.shape[-1]
valid_index, test_index, steps = VALID_INDEX, TEST_INDEX, PREDICT_STEPS
trade_dates = mask_data.shape[1]
learning_rate = 1e-3
market_states = 20
alpha = 0.1
epochs = 100
checkpoint_path = Path("stockmixer_best_valid.pt")


def build_offset_splits(valid_index, test_index, trade_dates, lookback=16, steps=1):
    """构造训练、验证、测试 offsets，保持与官方 train.py 一致。"""
    train_offsets = np.arange(start=0, stop=valid_index, dtype=int)
    train_steps = valid_index - lookback - steps + 1
    valid_offsets = list(range(valid_index - lookback - steps + 1, test_index - lookback - steps + 1))
    test_offsets = list(range(test_index - lookback - steps + 1, trade_dates - lookback - steps + 1))
    return train_offsets, train_steps, valid_offsets, test_offsets


def describe_training_plan(stocks, lookback, features, train_steps, valid_offsets, test_offsets, epochs):
    """用表格说明训练循环中每类 offset 的作用。"""
    return pd.DataFrame(
        [
            ["单步输入 x", (stocks, lookback, features), "一个交易日 offset 的全市场历史窗口"],
            ["训练步数/epoch", train_steps, "每个 epoch 使用的训练 offsets 数量"],
            ["验证 offsets", len(valid_offsets), "每个 epoch 后计算 val_loss 并选择 checkpoint"],
            ["测试 offsets", len(test_offsets), "训练结束后只用最佳 checkpoint 评估"],
            ["epochs", epochs, "官方配置，固定跑满，不做 early stopping"],
        ],
        columns=["项目", "值", "说明"],
    )


def train_one_epoch(model, optimizer, batch_offsets, train_steps_per_epoch, eod_data, price_data, gt_data, mask_data, lookback, steps, alpha):
    """训练一个 epoch，返回平均总 loss、回归 loss 和排序 loss。"""
    np.random.shuffle(batch_offsets)
    model.train()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_rank_loss = 0.0

    for j in range(train_steps_per_epoch):
        offset = int(batch_offsets[j])
        x, base_price, gt_return, valid_mask = get_batch(
            offset, eod_data, price_data, gt_data, mask_data, lookback, steps
        )
        optimizer.zero_grad()
        predicted_price = model(x)
        loss, reg_loss, rank_loss, _ = stockmixer_loss(
            predicted_price, gt_return, base_price, valid_mask, alpha=alpha
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_reg_loss += reg_loss.item()
        total_rank_loss += rank_loss.item()

    return {
        "loss": total_loss / train_steps_per_epoch,
        "reg_loss": total_reg_loss / train_steps_per_epoch,
        "rank_loss": total_rank_loss / train_steps_per_epoch,
    }


def save_training_checkpoint(path, epoch, model, optimizer, valid_result, config):
    """保存验证集 loss 最优的模型权重和复现实验所需配置。"""
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "valid_loss": valid_result["loss"],
            "valid_metrics": valid_result["metrics"],
            "config": config,
        },
        path,
    )


model = StockMixer(stocks=stocks, time_steps=lookback, channels=features, market_states=market_states).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

batch_offsets, train_steps_per_epoch, valid_offsets, test_offsets = build_offset_splits(
    valid_index, test_index, trade_dates, lookback, steps
)
training_config = {
    "stocks": stocks,
    "lookback": lookback,
    "features": features,
    "market_states": market_states,
    "alpha": alpha,
    "learning_rate": learning_rate,
    "valid_index": valid_index,
    "test_index": test_index,
    "steps": steps,
}

display(describe_training_plan(stocks, lookback, features, train_steps_per_epoch, valid_offsets, test_offsets, epochs))

history = {"train_loss": [], "train_reg_loss": [], "train_rank_loss": [], "val_loss": [], "best_val_loss": []}
best_val_loss = np.inf
best_epoch = -1

for epoch in range(epochs):
    train_result = train_one_epoch(
        model=model,
        optimizer=optimizer,
        batch_offsets=batch_offsets,
        train_steps_per_epoch=train_steps_per_epoch,
        eod_data=eod_data,
        price_data=price_data,
        gt_data=gt_data,
        mask_data=mask_data,
        lookback=lookback,
        steps=steps,
        alpha=alpha,
    )
    valid_result = validate_offsets(
        model, valid_offsets, eod_data, price_data, gt_data, mask_data, lookback, steps, alpha
    )
    val_loss = valid_result["loss"]

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        save_training_checkpoint(checkpoint_path, best_epoch, model, optimizer, valid_result, training_config)
        saved = " | saved best checkpoint"
    else:
        saved = ""

    history["train_loss"].append(train_result["loss"])
    history["train_reg_loss"].append(train_result["reg_loss"])
    history["train_rank_loss"].append(train_result["rank_loss"])
    history["val_loss"].append(val_loss)
    history["best_val_loss"].append(best_val_loss)

    print(
        f"epoch {epoch + 1:03d} | "
        f"train loss {train_result['loss']:.6f} "
        f"= {train_result['reg_loss']:.6f} + alpha*{train_result['rank_loss']:.6f} | "
        f"val loss {val_loss:.6f} | best epoch {best_epoch}{saved}"
    )


## 10. 测试与结果解释

测试阶段先加载验证集 loss 最低的 checkpoint，再按交易日逐日取横截面，得到形状 `(N, num_days)` 的预测收益矩阵和真实收益矩阵。这里的每一列对应一个交易日，每一行对应一只股票。

最终打印 5 个指标，与论文表格和官方 `evaluator.py` 的输出数量保持一致：`mse`、`IC`、`RIC`、`prec_10`、`sharpe5`。其中官方代码里的 `RIC` 是 `mean(IC) / std(IC)`，更接近 IC 稳定性比率；`sharpe5` 使用预测 top-5 股票的等权收益近似计算。


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
best_model = StockMixer(
    stocks=checkpoint["config"]["stocks"],
    time_steps=checkpoint["config"]["lookback"],
    channels=checkpoint["config"]["features"],
    market_states=checkpoint["config"]["market_states"],
).to(device)
best_model.load_state_dict(checkpoint["model_state_dict"])

valid_result = validate_offsets(
    best_model, valid_offsets, eod_data, price_data, gt_data, mask_data, lookback, steps, alpha
)
test_result = validate_offsets(
    best_model, test_offsets, eod_data, price_data, gt_data, mask_data, lookback, steps, alpha
)

pred_valid = valid_result["prediction"]
gt_valid = valid_result["ground_truth"]
mask_valid = valid_result["mask"]
pred_test = test_result["prediction"]
gt_test = test_result["ground_truth"]
mask_test = test_result["mask"]

print(f"Best checkpoint: {checkpoint_path}")
print(f"Best epoch: {checkpoint['epoch']}, valid loss: {checkpoint['valid_loss']:.6f}")
print("Valid performance:", format_stockmixer_metrics(valid_result["metrics"]))
print("Test performance :", format_stockmixer_metrics(test_result["metrics"]))

pd.Series(test_result["metrics"]).to_frame("test_value")


指标解读时要注意三点：

- 测试结果来自验证集最优 checkpoint，而不是最后一个 epoch 的权重；这对应量化研究中“用验证集选模型，用测试集做一次最终报告”的流程。
- 本 notebook 打印的测试指标数量为 5，与论文表格和官方 `evaluator.py` 一致；不要额外混入训练 loss 或验证 loss 作为测试指标。
- 在真实市场中，IC 很小也可能有意义。关键不是单日预测完全准确，而是长期横截面排序是否稳定、交易成本后是否仍有收益。


## 11. 可视化

先看训练 loss 是否下降，再看某一天的预测收益排序和真实收益排序。可视化的目的不是证明模型已经可交易，而是检查模型输出是否形成了可解释的横截面打分。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history["train_loss"], marker="o", label="train loss")
axes[0].plot(history["val_loss"], marker="o", label="valid loss")
axes[0].axvline(checkpoint["epoch"] - 1, color="tab:red", linestyle="--", linewidth=1.2, label="best valid")
axes[0].set_title("Training and validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

plot_day = 0
order = np.argsort(pred_valid[:, plot_day])
axes[1].scatter(np.arange(stocks), gt_valid[order, plot_day], s=18, label="true return")
axes[1].plot(np.arange(stocks), pred_valid[order, plot_day], color="tab:red", linewidth=1.5, label="predicted return")
axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.5)
axes[1].set_title("One validation day sorted by prediction")
axes[1].set_xlabel("Stocks sorted by predicted return")
axes[1].set_ylabel("Return")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


上图右侧按照模型预测收益从低到高排列股票。应重点观察：

- 右端高预测分数组中，真实收益是否更常为正
- 预测曲线是否出现极端异常值
- 真实收益点是否高度离散，提醒我们单日横截面噪声很大

如果在真实数据上长期看不到排序相关性，常见排查方向包括：标签是否错位、归一化是否泄漏未来、mask 是否正确、训练/验证/测试边界是否按交易日切分。


## 12. 小结

StockMixer 的核心价值不在于堆叠复杂结构，而是把股票预测中的三类相关性拆成轻量 mixing：

- indicator mixing 学习同一股票同一天的指标关系
- time mixing 用因果 TriU 和多尺度 patch 捕捉短窗口时间模式
- stock mixing 不依赖外部图，学习股票到市场状态再回到股票的横截面影响
- ranking-aware loss 让模型优化目标贴近选股排序，而不仅是价格回归

从量化验证角度，完整复现还需要固定数据版本、时间切分、交易假设、手续费滑点和多次随机种子。notebook 中的轻量示例只负责讲清楚张量如何流动，以及训练和评价闭环如何组织。

<details>
<summary>思考题：为什么不能随机打乱所有交易日后再划分训练集和测试集？</summary>

因为股票时间序列存在明显的时间依赖。随机划分会让训练集看到测试期附近甚至之后的市场状态，导致未来信息泄漏。量化实验应按交易日顺序划分，并在验证集上选择模型，最后只在测试集报告一次结果。

</details>

<details>
<summary>思考题：StockMixer 不使用行业图，是否意味着行业信息没有价值？</summary>

不是。StockMixer 的结论是：即使没有外部图，也可以通过 stock mixing 学习一部分市场共同状态。行业图、供应链图或新闻关系仍可能提供额外先验，但需要确认它们在样本外是否稳定、是否引入未来信息、是否覆盖所有股票。

</details>


## 13. 参考资料与延伸阅读

- Fan, Jinyong and Shen, Yanyan. **StockMixer: A Simple Yet Strong MLP-Based Architecture for Stock Price Forecasting**. AAAI 2024.
- 官方代码仓库：https://github.com/SJTU-Quant/StockMixer
- Feng et al. **Temporal Relational Ranking for Stock Prediction**，NASDAQ/NYSE 数据来源之一。
- Huynh et al. **Efficient Integration of Multi-Order Dynamics and Internal Dynamics in Stock Movement Prediction**，S&P500 数据来源之一。
- Tolstikhin et al. **MLP-Mixer: An all-MLP Architecture for Vision**，MLP mixing 思想来源之一。
- Zeng et al. **Are Transformers Effective for Time Series Forecasting?**，线性模型在时间序列预测中的代表性讨论。
